# Red Neuronal para discriminacion de Gamma/Neutron

Se creara una red neuronal que discrimine entre Gamma y Neutrones a partir de sus trazas de evento, esto a traves de Machine Learning implementado en la FPGA

*Código adaptado por:* Fabian Castaño

---

## Entrenamiento
Este entrenamiento se realizará basado en el trabajo previo:
- Workflow based on R. S. Molina, I. R. Morales, M. L. Crespo, V. G. Costa, S. Carrato and G. Ramponi, "An End-to-End Workflow to Efficiently Compress and Deploy DNN Classifiers on SoC/FPGA", in IEEE Embedded Systems Letters, vol. 16, no. 3, pp. 255-258, Sept. 2024, doi: 10.1109/LES.2023.3343030.
- Code adapted from the official repository of "An End-to-End Workflow to Efficiently Compress and Deploy DNN Classifiers on SoC/FPGA"
- Using open dataset from: https://doi.org/10.5281/zenodo.8037058


### Librerias

In [ ]:
import os
import numpy as np
from numpy import array
import matplotlib.pyplot as plt
import seaborn as sn
import pandas as pd

# Tensorflow and Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.models import *
from tensorflow.keras.layers import *
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.regularizers import l2, l1

import tensorflow_model_optimization as tfmot

# Pruning API
from tensorflow_model_optimization.python.core.sparsity.keras import prune, pruning_schedule, pruning_callbacks
from tensorflow_model_optimization.sparsity.keras import strip_pruning

# Quantization API
from qkeras import *

# Knowledge Distillation
from src.distillationClassKeras import *

# Metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils import shuffle

# Pre-processing
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder

# Training utils
from sklearn.model_selection import train_test_split

tf.random.set_seed(42)
np.random.seed(42)



In [ ]:
# Function to define the training and testing datasets

def preproc_dataset(signal_dfN, signal_dfG,
                    train_size=9000,
                    test_size=1000,
                    seed=42):

    LABEL = "class"

    # shuffle reproducible
    signal_dfN = shuffle(signal_dfN, random_state=seed)
    signal_dfG = shuffle(signal_dfG, random_state=seed)

    # seleccionar tamaños iguales
    train_neutron = signal_dfN.iloc[:train_size]
    test_neutron  = signal_dfN.iloc[train_size:train_size+test_size]

    train_gamma = signal_dfG.iloc[:train_size]
    test_gamma  = signal_dfG.iloc[train_size:train_size+test_size]

    # combinar
    dfTrain = pd.concat([train_neutron, train_gamma])
    dfTest  = pd.concat([test_neutron, test_gamma])

    # mezclar final
    dfTrain = shuffle(dfTrain, random_state=seed).reset_index(drop=True)
    dfTest  = shuffle(dfTest, random_state=seed).reset_index(drop=True)

    # ---------- NORMALIZACION (CRITICO PARA HLS) ----------

    features = dfTrain.columns[:-1]

    dfTrain[features] = dfTrain[features] / 512.0
    dfTest[features]  = dfTest[features] / 512.0

    return dfTrain, dfTest


### Carga de datos

In [ ]:
# Define paths
PATH = "dataset/"

TRAIN_DATASET_FILE = PATH + "train_dataset.csv"
TEST_DATASET_FILE  = PATH + "test_dataset.csv"

# Load datasets
dfTrain = pd.read_csv(TRAIN_DATASET_FILE)
dfTestGN = pd.read_csv(TEST_DATASET_FILE)

# Separate classes
dfGamma = dfTrain[dfTrain["class"] == 0]
dfNeutron = dfTrain[dfTrain["class"] == 1]

### Preprocesamiento y separacion de dataset

Separacion del dataset en datos de entrenamiento y de test

In [ ]:
# Pre-processing dataset for training
df_train, dfTest = preproc_dataset(dfNeutron, dfGamma)

# Save the test
# dfTest.to_csv('dataset/test.csv')

df_train_ = df_train.pop('class')
dfTest_   = dfTest.pop('class')

# One-hot encoder
le = LabelEncoder()
y = le.fit_transform(df_train_)
y = to_categorical(y, 2)

le = LabelEncoder()
yTest = le.fit_transform(dfTest_)
yTest = to_categorical(yTest, 2)

# Split training dataset into training and validation
x_train, x_val, y_train, y_val = train_test_split(df_train, y, test_size=0.3, random_state=42)

x_train = x_train.astype(np.float32)
x_val   = x_val.astype(np.float32)



## Carga del modelo

Aqui se carga el modelo y los datos de entrenamiento

In [ ]:
teacher_model = keras.models.load_model("models/teacherModel_GN_GICM.h5")
teacher_model.summary()

### Podado, Pruning

Aqui se hara el proceso de podado y reduccion de complejidad del modelo para el modelo entrenado. Primero se cargara el modelo previamente guardado

![Distilation](images/distillation.png)

In [ ]:
epochs = 100
lr = 0.001
loss = 'categorical_crossentropy'
op = Adam(learning_rate=lr)
metrics = ['accuracy']
batch = 64
val_split = 0.2

final_sparsity = 0.3 # percentage of weights pruned

pruning_params = {
                 'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
                    initial_sparsity=0, 
                    final_sparsity=final_sparsity, 
                    begin_step=0, 
                    end_step=1000
                 )
                 }

In [ ]:
callbacks = [
    # tf.keras.callbacks.EarlyStopping(
    #     monitor='val_loss',
    #     patience=2,
    #     verbose=1,
    #     restore_best_weights=True
    # ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', #'val_loss',
        factor=0.4,
        patience=3,
        verbose=1
    ), 
    pruning_callbacks.UpdatePruningStep()
]

### Podado del modelo

In [ ]:
modelP = tfmot.sparsity.keras.prune_low_magnitude(teacher_model, **pruning_params)
modelP.compile(optimizer=op, loss=loss, metrics=metrics)

In [ ]:
history_P = modelP.fit(x=x_train, y=y_train,
                       validation_split=val_split,
                       batch_size=batch, 
                       epochs=epochs,
                       callbacks= callbacks,
                       verbose=1
                       )

In [ ]:
print(history_P.history.keys())

## Plot for accuracy
plt.figure(figsize=(15,3))
plt.plot(history_P.history['accuracy'])
plt.plot(history_P.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()
plt.figure()

## Plot for loss
plt.figure(figsize=(15,3))
plt.plot(history_P.history['loss'])
plt.plot(history_P.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

### Confusion matrix

Matriz de confusion despues del podado

In [ ]:
# Obtain the confusion matrix using the testing dataset 
y_pred_probs = modelP.predict(dfTest)
y_pred = np.argmax(y_pred_probs, axis=1)

# Since y_test is one-hot encoded, you need to convert it back to class indices
y_true = np.argmax(yTest, axis=1)  # Convert one-hot encoded labels to class indices

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title('Confusion Matrix - Teacher model after pruning')
plt.show()

### Paso obligatorio después del pruning

Antes de exportar el modelo debes eliminar los wrappers:

In [ ]:
modelP = strip_pruning(modelP)

---

## Cuantizacion (QAT)

Aqui se aplicara la cuantizacion para reducir el tamaño y resolucion de la red

In [ ]:
## Estrategia de cuantizacion

## Definicion del numero de bits para kernel, bias, y activacion.
# 8-bits

neurons_teacher = [10, 5, 7, 5, 6]

kernelQ      = "quantized_bits(8,4,alpha=1)"
biasQ        = "quantized_bits(8,4,alpha=1)"
activationQ  = "quantized_bits(8,4,alpha=1)"

modelQAT = Sequential(
    [
        QDense(neurons_teacher[0], name='input', input_shape=(161,),
                           kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                           kernel_initializer='lecun_uniform'),
                    QActivation(activation=activationQ, name='relu0'),
                    # Dropout(0.1),

                    QDense(neurons_teacher[1], name='fc1', 
                           kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                           kernel_initializer='lecun_uniform'),
                    QActivation(activation=activationQ, name='relu1'),
                    # Dropout(0.1),

                    QDense(neurons_teacher[2], name='fc2',
                           kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                           kernel_initializer='lecun_uniform'),
                    QActivation(activation=activationQ, name='relu2'),
                    Dropout(0.1),

                    QDense(neurons_teacher[3], name='fc3',
                           kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                           kernel_initializer='lecun_uniform'),
                    QActivation(activation=activationQ, name='relu3'), 
                    Dropout(0.2),
                
                    QDense(neurons_teacher[4], name='fc4',
                           kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                           kernel_initializer='lecun_uniform'),
                    QActivation(activation=activationQ, name='relu4'), 
                    Dropout(0.2),
                
                    QDense(2, name='output',
                           kernel_quantizer= kernelQ, bias_quantizer= biasQ,
                           kernel_initializer='lecun_uniform'),
                    Activation(activation='sigmoid', name='sigmoid')       
    ], 
    name = "QuantizedModel",
)

In [ ]:
modelQAT.summary()

### Creacion del modelo cuantizado

Aqui se realiza el entrenamiento

In [ ]:
callbacks = [
    # tf.keras.callbacks.EarlyStopping(
    #     monitor='val_loss',
    #     patience=2,
    #     verbose=1,
    #     restore_best_weights=True
    # ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', #'val_loss',
        factor=0.4,
        patience=3,
        verbose=1
    ), 
    pruning_callbacks.UpdatePruningStep()
]

In [ ]:
epochs = 150
lr = 0.0001
loss = 'binary_crossentropy'
op = Adam(learning_rate=lr)
metrics = ['accuracy']
batch = 32  
val_split = 0.2

modelQAT.compile(optimizer=op, loss=loss, metrics=metrics)

history_QAT = modelQAT.fit(x=x_train, y=y_train,
                       validation_split=val_split,
                       epochs=epochs,
                       batch_size=batch,
                       callbacks= callbacks,
                       verbose=1
                       )


In [ ]:
print(history_QAT.history.keys())

## Plot for accuracy
plt.figure(figsize=(15,3))
plt.plot(history_QAT.history['accuracy'])
plt.plot(history_QAT.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()
plt.figure()

## Plot for loss
plt.figure(figsize=(15,3))
plt.plot(history_QAT.history['loss'])
plt.plot(history_QAT.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

In [ ]:
# Obtain the confusion matrix using the testing dataset 
y_pred_probs = modelQAT.predict(dfTest)
y_pred = np.argmax(y_pred_probs, axis=1)

# Since y_test is one-hot encoded, you need to convert it back to class indices
y_true = np.argmax(yTest, axis=1)  # Convert one-hot encoded labels to class indices

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title('Confusion Matrix - Teacher model obtained through quantization aware training')
plt.show()

### Guardado del modelo podado y cuantizado

In [ ]:
# If needed, save the teacher QAT model [uncomment the following line]
modelQAT.save('models/teacherQATModel_GN_GICM.h5')

---

# Entrenamiento del modelo Student

Para obtener el modelo reducido que será implementado en la FPGA (llamado en adelante red estudiante), se empleará un enfoque de aprendizaje por destilación de conocimiento (**KD**).

En este método, el modelo maestro transfiere lo aprendido al modelo estudiante.

El modelo estudiante se define aplicando cuantización y poda.

En este proyecto se fijaron:

- 8 bits para la cuantización
- 50% de sparsity (poda de la mitad de los pesos).

Para la cuantización se utiliza QKeras, una librería especializada para redes neuronales cuantizadas y optimizadas para hardware.

**Referencia sobre QKeras**:
Coelho et al. (2020). Ultra low-latency, low-area inference accelerators using heterogeneous deep quantization with QKeras and hls4ml. arXiv:2006.10159.

### Hiperparametros para el modelo Student

![Distilation](images/studentArchitecture.png)

In [ ]:
# Define the hyperparameters for the student model
lr = 0.001
neurons_student = [6, 4, 2, 4, 3]

In [ ]:
def student_topology(neurons_student):
    
    '''
    Model to be trained. Defined with quantization strategies. 
    Input: hyperparams. (bestHP)
    Output: compressed model (studentQ_MLP). 

    '''
    ######## ---------------------------  Model definition - 1D STUDENT -----------------------------------------

    # Number of bits 
    ## 4-bits
    kernelQ_4b = "quantized_bits(4,2,alpha=1)"
    biasQ_4b = "quantized_bits(4,2,alpha=1)"
    activationQ_4b = 'quantized_bits(4, 0)'

    ## 8-bits
    kernelQ = "quantized_bits(8,4,alpha=1)"
    biasQ = "quantized_bits(8,4,alpha=1)"
    activationQ = 'quantized_bits(8,4)'
    
    ## 16-bits
    kernelQ_16b = "quantized_bits(16,6,alpha=1)"
    biasQ_16b = "quantized_bits(16,6,alpha=1)"
    activationQ_16b = 'quantized_bits(16,6)'
    
    studentQ_MLP = keras.Sequential(
            [   
                Input(shape=(161,), name='inputLayer'),
                QDense(neurons_student[0], name='fc1',
                        kernel_quantizer= kernelQ, bias_quantizer= biasQ,
                        kernel_initializer='lecun_uniform'),
                QActivation(activation= activationQ ,  name='relu0'),
                Dropout(0.1),
                    
                QDense(neurons_student[1], name='fc2',
                        kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                        kernel_initializer='lecun_uniform'),
                QActivation(activation=activationQ, name='relu1'), 
                Dropout(0.1),
                

                QDense(neurons_student[2], name='fc3',
                        kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                        kernel_initializer='lecun_uniform'),
                QActivation(activation=activationQ, name='relu2'), 
                Dropout(0.1),

                QDense(neurons_student[3], name='fc4',
                        kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                        kernel_initializer='lecun_uniform'),
                QActivation(activation=activationQ, name='relu3'), 
                Dropout(0.2),

                QDense(neurons_student[4], name='fc5',
                       kernel_quantizer=kernelQ, bias_quantizer=biasQ,
                       kernel_initializer='lecun_uniform'),
                QActivation(activation=activationQ, name='relu4'), 
                Dropout(0.2),
               
                QDense(2, name='output',
                        kernel_quantizer= kernelQ_16b, bias_quantizer= biasQ_16b,
                        kernel_initializer='lecun_uniform'),
                Activation(activation='sigmoid', name='outputActivation')

                
            ],
            name="studentMLP",
        )


    return studentQ_MLP

### Construccion del modelo

In [ ]:
# Build student model with pruning strategy

def build_student(student_neurons, x_train, batch_size=64, epochs=200):

    '''
    Build pruned student model
    '''

    qmodel = student_topology(student_neurons)

    steps_per_epoch = np.ceil(len(x_train) / batch_size)
    end_step = steps_per_epoch * epochs

    pruning_params = {
        "pruning_schedule": tfmot.sparsity.keras.PolynomialDecay(
            initial_sparsity=0.0,
            final_sparsity=0.5,
            begin_step=0,
            end_step=end_step
        )
    }

    studentQ = tfmot.sparsity.keras.prune_low_magnitude(
        qmodel,
        **pruning_params
    )

    return studentQ

---

### Compilacion del modelo Student para KD y QAT

Compilacion para destilacion de conocimiento y cuantizacion

In [ ]:
studentQ = build_student(neurons_student, x_train)
studentQ.summary()

In [ ]:
distilled_student = Distiller(student=studentQ, teacher=modelQAT)
adam = Adam(learning_rate=lr)
train_labels = np.argmax(y_train, axis=1)

distilled_student.compile(
        optimizer=adam,
        metrics=[keras.metrics.SparseCategoricalAccuracy()],
        student_loss_fn=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        distillation_loss_fn=keras.losses.KLDivergence(),
        alpha=0.1,
        temperature=10,
    )

### Entrenamiento del modelo Student

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_student_loss',
        patience=10,
        verbose=1,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_student_loss', #'val_loss',
        factor=0.3,
        patience=3,
        verbose=1
    ), 
    pruning_callbacks.UpdatePruningStep(),
]

history_student = distilled_student.fit(
    x=x_train,
    y=train_labels,
    batch_size=64,
    epochs=200,
    validation_split=0.2,
    callbacks=callbacks,
)

In [ ]:
print(history_student.history.keys())

## Plot for accuracy
plt.figure(figsize=(15,3))
plt.plot(history_student.history['sparse_categorical_accuracy'])
plt.plot(history_student.history['val_sparse_categorical_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()
plt.figure()

## Plot for loss
plt.figure(figsize=(15,3))
plt.plot(history_student.history['student_loss'])
plt.plot(history_student.history['val_student_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

In [ ]:
# Obtain the confusion matrix using the testing dataset 
y_pred_probs = distilled_student.student.predict(dfTest)
y_pred = np.argmax(y_pred_probs, axis=1)

# Since y_test is one-hot encoded, you need to convert it back to class indices
y_true = np.argmax(yTest, axis=1)  # Convert one-hot encoded labels to class indices

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title('Confusion Matrix - Student model Q, P, and KD')
plt.show()

### Prueba de predicción individual

Prueba el modelo con entradas individuales.

La variable indexPrediction corresponde al índice de la señal en el conjunto de datos de prueba. Puedes cambiar este número para observar cómo el modelo basado en ML predice para entradas específicas.

In [ ]:
# Index of the signal in the testing dataset
indexPrediction = 405

x_input = dfTest.iloc[indexPrediction]
y_label = yTest[indexPrediction]

inputPred = array([x_input])

y_pred = distilled_student.student.predict(inputPred) 
print("> Input %s -> Predicted = %s | " % (y_label, y_pred))

plt.figure(figsize=(15,3))
plt.xlabel('Samples', fontsize=11)
plt.ylabel('Amplitude', fontsize=11)
plt.grid(True, alpha=1.0)
plt.plot(x_input.values,  label="Signal 1", color='navy', markersize=7, lw=1)

plt.title('Signal used for inference - Pulse: %s, Label: %s, Prediction: %s' % (indexPrediction, y_label, y_pred))

In [ ]:
# Index of the signal in the testing dataset
indexPrediction = 200

x_input = dfTest.iloc[indexPrediction]
y_label = yTest[indexPrediction]

inputPred = array([x_input])

y_pred = distilled_student.student.predict(inputPred) 
print("> Input %s -> Predicted = %s | " % (y_label, y_pred))

plt.figure(figsize=(15,3))
plt.xlabel('Samples', fontsize=11)
plt.ylabel('Amplitude', fontsize=11)
plt.grid(True, alpha=1.0)
plt.plot(x_input.values,  label="Signal 1", color='navy', markersize=7, lw=1)

plt.title('Signal used for inference - Pulse: %s, Label: %s, Prediction: %s' % (indexPrediction, y_label, y_pred))

### Guardar el modelo Student

In [ ]:
model_student = strip_pruning(distilled_student.student)
model_student.summary()

model_student.save('models/studentModel_GN_GICM.h5')

---
**Autor**: Fabian Castaño